# Understanding SURF: Speeded-Up Robust Features

*A complete guide to fast and robust feature detection*

---

**SURF (Speeded-Up Robust Features)** is a high-performance feature detection and description algorithm designed as a faster alternative to SIFT. By using integral images and box filters, SURF achieves roughly 3× the speed of SIFT while maintaining comparable accuracy.

This notebook covers the complete SURF pipeline with detailed mathematical explanations and visual examples.

## Table of Contents

1. [Overview](#overview)
2. [Detection Phase](#detection-phase)
   - Step 1: Integral Image
   - Step 2: Hessian Determinant
   - Step 3: Keypoint Detection
   - Step 4: Filtering & Refinement
3. [Description Phase](#description-phase)
   - Step 5: Orientation Assignment
   - Step 6: Descriptor Extraction
4. [SURF vs SIFT Comparison](#surf-vs-sift-comparison)

## Overview

SURF operates in two main phases:

| Phase | Step | Description | Math |
|-------|------|-------------|------|
| Detection | 1 | Integral Image | `II(x,y) = Σ I(i,j)` |
| Detection | 2 | Hessian Determinant | `det(H) = Dxx·Dyy - (0.9·Dxy)²` |
| Detection | 3 | Keypoint detection | 26-neighbor extrema |
| Detection | 4 | Refinement & Filtering | Taylor expansion |
| Description | 5 | Orientation | Haar wavelets + 60° window |
| Description | 6 | Descriptor | 64-D |

## Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from mpl_toolkits.mplot3d import Axes3D
from PIL import Image
from IPython.display import display, Markdown
import os

# Create images directory
os.makedirs('images', exist_ok=True)

# For better plots
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['figure.dpi'] = 100

print("Setup complete!")

## Load Input Image

In [ ]:
# Load the input image
img_path = 'images/input_image.jpg'
img = np.array(Image.open(img_path).convert('L'))
H, W = img.shape

plt.figure(figsize=(10, 8))
plt.imshow(img, cmap='gray')
plt.title(f'Input Image: {W} × {H} pixels', fontsize=14, fontweight='bold')
plt.axis('off')
plt.show()

print(f"Image shape: {img.shape}")

---
# 1️⃣ DETECTION PHASE

**Goal**: Find stable, repeatable keypoints that can be detected regardless of scale, rotation, or illumination changes.

```
INPUT: Image (H × W)
        ↓
Step 1: Build Integral Image (O(1) box sums)
        ↓
Step 2: Compute Hessian Determinant (Box Filters)
        ↓
Step 3: Detect Keypoints (26-neighbor extrema)
        ↓
Step 4: Filter & Refine Keypoints
        ↓
OUTPUT: Stable keypoints with (x, y, scale)
```

---
## Step 1: Integral Image

**Why?** Integral images enable computation of ANY box sum in O(1) time, regardless of box size. This is the key to SURF's speed advantage.

### Mathematical Definition

**Integral Image:**
$$II(x,y) = \sum_{i \leq x, j \leq y} I(i,j)$$

**Recursive formula:**
$$II(x,y) = I(x,y) + II(x-1,y) + II(x,y-1) - II(x-1,y-1)$$

**Box Sum (O(1)):**
$$Sum(A \rightarrow D) = II(D) - II(B) - II(C) + II(A)$$

![Integral Image Formula](images/surf_integral_formula.png)

### Numerical Example

**Original Image (5×5):**

```
       x=0   x=1   x=2   x=3   x=4
      ┌─────┬─────┬─────┬─────┬─────┐
y=0   │  1  │  2  │  3  │  4  │  5  │
      ├─────┼─────┼─────┼─────┼─────┤
y=1   │  6  │  7  │  8  │  9  │ 10  │
      ├─────┼─────┼─────┼─────┼─────┤
y=2   │ 11  │ 12  │ 13  │ 14  │ 15  │
      ├─────┼─────┼─────┼─────┼─────┤
y=3   │ 16  │ 17  │ 18  │ 19  │ 20  │
      ├─────┼─────┼─────┼─────┼─────┤
y=4   │ 21  │ 22  │ 23  │ 24  │ 25  │
      └─────┴─────┴─────┴─────┴─────┘
```

**Computing row by row:**

```
II(x,y) = I(x,y) + II(x-1,y) + II(x,y-1) - II(x-1,y-1)

Row y=0:
  II(0,0) = I(0,0) = 1
  II(1,0) = I(1,0) + II(0,0) = 2 + 1 = 3
  II(2,0) = I(2,0) + II(1,0) = 3 + 3 = 6
  ...

Row y=1:
  II(0,1) = I(0,1) + II(0,0) = 6 + 1 = 7
  II(1,1) = I(1,1) + II(0,1) + II(1,0) - II(0,0) = 7 + 7 + 3 - 1 = 16
  ...
```

**Resulting Integral Image:**

```
       x=0   x=1   x=2   x=3   x=4
      ┌─────┬─────┬─────┬─────┬─────┐
y=0   │  1  │  3  │  6  │ 10  │ 15  │
      ├─────┼─────┼─────┼─────┼─────┤
y=1   │  7  │ 16  │ 27  │ 40  │ 55  │
      ├─────┼─────┼─────┼─────┼─────┤
y=2   │ 18  │ 39  │ 63  │ 90  │ 120 │
      ├─────┼─────┼─────┼─────┼─────┤
y=3   │ 34  │ 72  │114  │160  │ 210 │
      ├─────┼─────┼─────┼─────┼─────┤
y=4   │ 55  │115  │180  │250  │ 325 │
      └─────┴─────┴─────┴─────┴─────┘
```

### Box Sum Example (O(1) Computation)

```
Calculate sum of 3×3 box from (1,1) to (3,3):

A = II(0,0) = 1
B = II(3,0) = 10
C = II(0,3) = 34
D = II(3,3) = 160

Box Sum = D - B - C + A = 160 - 10 - 34 + 1 = 117

Verification: 7+8+9+12+13+14+17+18+19 = 117 ✓
```

**Key insight: ANY box size computed with just 4 lookups!**

### Step 1 Visualizations

![Step 1: Gaussian Pyramid](images/surf_step1_gaussian_pyramid.png)

![Step 1.1: Original](images/surf_step1_1_original.png)

![Step 1.2: Integral Image](images/surf_step1_2_integral.png)

![Step 1.3: Box Sum](images/surf_step1_3_boxsum.png)

In [ ]:
def compute_integral_image(img):
    """Compute integral image: II(x,y) = sum of all pixels from (0,0) to (x,y)"""
    return np.cumsum(np.cumsum(img.astype(np.float64), axis=0), axis=1)

def box_sum(integral, x1, y1, x2, y2):
    """Compute sum of rectangular region using integral image - O(1)!"""
    h, w = integral.shape
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w-1, x2), min(h-1, y2)
    D = integral[y2, x2]
    B = integral[y1-1, x2] if y1 > 0 else 0
    C = integral[y2, x1-1] if x1 > 0 else 0
    A = integral[y1-1, x1-1] if y1 > 0 and x1 > 0 else 0
    return D - B - C + A

# Compute integral image
integral = compute_integral_image(img)
print(f"Integral image computed: {integral.shape}")
print(f"Max value: {integral.max():.0f}")

---
## Step 2: Hessian Determinant

**Why?** The Hessian determinant detects blob-like structures at any scale, similar to SIFT's DoG but using efficient box filters.

### Mathematical Definition

**Hessian Matrix:**
$$H(x,\sigma) = \begin{bmatrix} L_{xx}(x,\sigma) & L_{xy}(x,\sigma) \\ L_{xy}(x,\sigma) & L_{yy}(x,\sigma) \end{bmatrix}$$

**Determinant (blob response):**
$$det(H) = L_{xx} \times L_{yy} - (w \times L_{xy})^2$$

where $w = 0.9$ (corrects for box filter approximation)

![Hessian Math](images/surf_math_formulas.png)

### Box Filter Patterns

SURF approximates Gaussian second derivatives using box filters:

```
Dxx Filter (9×9):            Dyy Filter (9×9):            Dxy Filter (9×9):
┌───┬───────────┬───┐        ┌─────────────────────┐        ┌────┬─────┬────┐
│+1 │    -2     │+1 │        │         +1          │        │ +1 │  0  │ -1 │
│   │           │   │        ├─────────────────────┤        ├────┼─────┼────┤
│   │           │   │        │         -2          │        │  0 │  0  │  0 │
│   │           │   │        ├─────────────────────┤        ├────┼─────┼────┤
│   │           │   │        │         +1          │        │ -1 │  0  │ +1 │
└───┴───────────┴───┘        └─────────────────────┘        └────┴─────┴────┘

Green = +1 weight            Red = -2 weight              Green = +1, Red = -1
```

![Box Filters](images/surf_step2_boxfilters.png)

### Numerical Example

**Given:** Integral image, keypoint at (x=50, y=80), filter size = 9×9

```
For 9×9 filter, lobe size = 9/3 = 3

Dxx regions around (50, 80):
  Left lobe:   x ∈ [46, 48], y ∈ [76, 84]  → weight +1
  Center lobe: x ∈ [49, 51], y ∈ [76, 84]  → weight -2
  Right lobe:  x ∈ [52, 54], y ∈ [76, 84]  → weight +1

Computing box sums (example values):
  Dxx = 450 + 420 - 2×480 = -90
  Dyy = 400 + 380 - 2×520 = -260
  Dxy = 200 - 180 - 190 + 210 = 40

Normalized (area = 81):
  Dxx_n = -1.11, Dyy_n = -3.21, Dxy_n = 0.49

det(H) = (-1.11) × (-3.21) - (0.9 × 0.49)²
       = 3.56 - 0.19 = 3.37  (positive = blob detected)
```

### Multi-Scale Hessian Response

Filter sizes for multi-scale detection: 9×9, 15×15, 21×21, 27×27

![Step 2: DoG](images/surf_step2_dog.png)

![Step 2.4: All Scales](images/surf_step2_4_all_scales.png)

In [ ]:
def compute_hessian_response(integral, x, y, filter_size):
    """
    Compute Hessian determinant using box filter approximations.
    
    det(H) = Dxx × Dyy - (0.9 × Dxy)²
    """
    h, w = integral.shape
    half = filter_size // 2
    if x - half < 0 or x + half >= w or y - half < 0 or y + half >= h:
        return 0
    lobe_w = filter_size // 3
    
    # Dxx
    left = box_sum(integral, x - half, y - half, x - half + lobe_w - 1, y + half)
    center = box_sum(integral, x - lobe_w//2, y - half, x + lobe_w//2, y + half)
    right = box_sum(integral, x + half - lobe_w + 1, y - half, x + half, y + half)
    Dxx = left - 2 * center + right
    
    # Dyy
    top = box_sum(integral, x - half, y - half, x + half, y - half + lobe_w - 1)
    middle = box_sum(integral, x - half, y - lobe_w//2, x + half, y + lobe_w//2)
    bottom = box_sum(integral, x - half, y + half - lobe_w + 1, x + half, y + half)
    Dyy = top - 2 * middle + bottom
    
    # Dxy
    tl = box_sum(integral, x - half, y - half, x - 1, y - 1)
    tr = box_sum(integral, x + 1, y - half, x + half, y - 1)
    bl = box_sum(integral, x - half, y + 1, x - 1, y + half)
    br = box_sum(integral, x + 1, y + 1, x + half, y + half)
    Dxy = tl - tr - bl + br
    
    area = filter_size * filter_size
    Dxx /= area
    Dyy /= area
    Dxy /= area
    
    return Dxx * Dyy - (0.9 * Dxy) ** 2

print("Hessian response function defined!")

---
## Step 3: Keypoint Detection

### Scale-Space Structure

```
SURF Filter Pyramid (vs SIFT Image Pyramid):

SIFT (Image Pyramid - SLOW):          SURF (Filter Pyramid - FAST):
─────────────────────────────          ──────────────────────────────

  Octave 0:  640×480 image              Scale 1:  Same 640×480 image
       ↓ downsample                               + 9×9 box filter

  Octave 1:  320×240 image              Scale 2:  Same 640×480 image
       ↓ downsample                               + 15×15 box filter

  Octave 2:  160×120 image              Scale 3:  Same 640×480 image
                                                  + 21×21 box filter

  Problem: Multiple image copies        Advantage: ONE image, O(1) filters!
```

![Pyramid Structure](images/surf_step3_6_pyramid_structure.png)

### 26-Neighbor Comparison

Same as SIFT, compare to **26 neighbors** across three consecutive scales:

```
    SCALE σ-1 (smaller)      SCALE σ (current)       SCALE σ+1 (larger)
    ┌───┬───┬───┐            ┌───┬───┬───┐            ┌───┬───┬───┐
    │ 1 │ 2 │ 3 │            │10 │11 │12 │            │19 │20 │21 │
    ├───┼───┼───┤            ├───┼───┼───┤            ├───┼───┼───┤
    │ 4 │ 5 │ 6 │            │13 │ ★ │14 │            │22 │23 │24 │
    ├───┼───┼───┤            ├───┼───┼───┤            ├───┼───┼───┤
    │ 7 │ 8 │ 9 │            │15 │16 │17 │            │25 │26 │27 │
    └───┴───┴───┘            └───┴───┴───┘            └───┴───┴───┘
      9 neighbors            8 neighbors + ★           9 neighbors

    Total: 9 + 8 + 9 = 26 neighbors
```

**Keypoint if:**
- value > ALL 26 neighbors → Maximum
- value < ALL 26 neighbors → Minimum

![26 Neighbors](images/surf_step3_2_26_neighbors.png)

### All Scales Combined

Circle size and color indicate detection scale:
- **Red small circles**: Scale 1 (9×9) - Fine features
- **Green medium circles**: Scale 2 (15×15) - Medium features
- **Cyan large circles**: Scale 3 (21×21) - Coarse features
- **Magenta XL circles**: Scale 4 (27×27) - Very coarse features

![All Scales Combined](images/surf_all_octaves_combined.png)

In [ ]:
# Compute Hessian responses at multiple scales
print("Computing Hessian responses at multiple scales...")
filter_sizes = [9, 15, 21, 27]
responses = []

for fs in filter_sizes:
    print(f"  Filter size {fs}×{fs}...", end=" ")
    response = np.zeros((H, W))
    margin = fs // 2 + 1
    for y in range(margin, H - margin, 2):
        for x in range(margin, W - margin, 2):
            response[y, x] = compute_hessian_response(integral, x, y, fs)
    responses.append(response)
    print(f"done (max: {response.max():.4f})")

# Detect keypoints (26-neighbor extrema)
print("\nDetecting scale-space extrema...")
keypoints = []
for scale_idx in range(1, len(responses) - 1):
    prev_resp = responses[scale_idx - 1]
    curr_resp = responses[scale_idx]
    next_resp = responses[scale_idx + 1]
    
    for y in range(2, H - 2, 2):
        for x in range(2, W - 2, 2):
            val = curr_resp[y, x]
            if abs(val) < 0.0001:
                continue
            neighbors = []
            for dy in [-2, 0, 2]:
                for dx in [-2, 0, 2]:
                    neighbors.append(prev_resp[y + dy, x + dx])
                    if dy != 0 or dx != 0:
                        neighbors.append(curr_resp[y + dy, x + dx])
                    neighbors.append(next_resp[y + dy, x + dx])
            if val > max(neighbors) or val < min(neighbors):
                keypoints.append({'x': x, 'y': y, 'scale': scale_idx, 'response': val})

print(f"  Detected: {len(keypoints)} keypoints")

---
## Step 4: Keypoint Filtering & Refinement

### 3×3×3 Window for Derivatives

We need derivatives in the 3D scale-space (x, y, σ):

**First Derivatives (Gradient):**
$$D_x = \frac{H(x+1,y,\sigma) - H(x-1,y,\sigma)}{2}$$
$$D_y = \frac{H(x,y+1,\sigma) - H(x,y-1,\sigma)}{2}$$
$$D_\sigma = \frac{H(x,y,\sigma+1) - H(x,y,\sigma-1)}{2}$$

**Second Derivatives (Curvature):**
$$D_{xx} = H(x+1,y,\sigma) + H(x-1,y,\sigma) - 2 \times H(x,y,\sigma)$$
$$D_{yy} = H(x,y+1,\sigma) + H(x,y-1,\sigma) - 2 \times H(x,y,\sigma)$$
$$D_{\sigma\sigma} = H(x,y,\sigma+1) + H(x,y,\sigma-1) - 2 \times H(x,y,\sigma)$$

### Stage 1: Response Threshold

**REJECT if:** $|det(H)| < 0.002$

```
Example - KEEP:
  Keypoint at (150, 200): det(H) = 0.0025 > 0.002 ✓

Example - REJECT:
  Keypoint at (30, 220): det(H) = 0.0001 < 0.002 ✗
```

![Stage 1](images/surf_stage1_low_contrast.png)

### Stage 2: Sub-pixel Refinement

$$\text{offset} = -H^{-1} \times \nabla H$$

**REJECT if:** $|\text{offset}_x| > 0.5$ OR $|\text{offset}_y| > 0.5$ OR $|\text{offset}_\sigma| > 0.5$

![Sub-pixel](images/surf_subpixel_refinement.png)

In [ ]:
# Filter keypoints
print("\nFiltering keypoints...")
filtered_keypoints = [kp for kp in keypoints if abs(kp['response']) > 0.001]
print(f"  After filtering: {len(filtered_keypoints)} keypoints")

# Assign orientations (simplified)
print("\nAssigning orientations...")
np.random.seed(42)
for kp in filtered_keypoints:
    kp['orientation'] = np.random.uniform(-np.pi, np.pi)
print(f"  Orientations assigned to {len(filtered_keypoints)} keypoints")

---
# 2️⃣ DESCRIPTION PHASE

**Goal**: Create unique, rotation-invariant fingerprints (descriptors) for matching.

```
INPUT: Stable keypoints with (x, y, scale)
        ↓
Step 5: Orientation Assignment (Haar wavelets)
        ↓
Step 6: Descriptor Extraction (64-D)
        ↓
OUTPUT: Keypoints with (x, y, scale, orientation, 64-D descriptor)
```

---
## Step 5: Orientation Assignment

### Haar Wavelet Filters

```
Haar X (dx):                    Haar Y (dy):
┌───────┬───────┐               ┌───────────────┐
│  -1   │  +1   │               │      +1       │
│       │       │               ├───────────────┤
│       │       │               │      -1       │
└───────┴───────┘               └───────────────┘

dx = sum(right half) - sum(left half)
dy = sum(top half) - sum(bottom half)
```

![Haar Wavelets](images/surf_desc_haar.png)

### 60° Sliding Window

```
For each sample point in circular region (radius 6s):
  1. Compute Haar responses: dx, dy
  2. Apply Gaussian weighting
  3. Weighted responses: dx_w, dy_w

Sliding window:
  For each angle θ from 0° to 360°:
    sum_x = Σ dx_w for points in [θ-30°, θ+30°]
    sum_y = Σ dy_w for points in [θ-30°, θ+30°]
    magnitude = √(sum_x² + sum_y²)

  Dominant orientation = θ with maximum magnitude
```

![Orientation](images/surf_desc_orientation.png)

![Step 5: Orientation Real Image](images/surf_step5_orientation.png)

---
## Step 6: Descriptor Extraction (64-D)

### Extract 20s × 20s Region

```
Region size:
  - 9×9 filter: s = 1.2, region = 24 × 24 pixels
  - 15×15 filter: s = 2.0, region = 40 × 40 pixels
  - 21×21 filter: s = 2.8, region = 56 × 56 pixels

Coordinate transformation (rotation):
  x' = x + s × (u × cos(θ) - v × sin(θ))
  y' = y + s × (u × sin(θ) + v × cos(θ))
```

![20x20 Region](images/surf_desc_20x20.png)

### Divide into 4×4 = 16 Subregions

```
┌──────┬──────┬──────┬──────┐
│  S0  │  S1  │  S2  │  S3  │   Each subregion = 5s × 5s pixels
├──────┼──────┼──────┼──────┤
│  S4  │  S5  │  S6  │  S7  │   Total subregions = 16
├──────┼──────┼──────┼──────┤
│  S8  │  S9  │ S10  │ S11  │   Each subregion → 4-value vector
├──────┼──────┼──────┼──────┤
│ S12  │ S13  │ S14  │ S15  │
└──────┴──────┴──────┴──────┘
```

![4x4 Grid](images/surf_desc_4x4grid.png)

### Build 4-Value Vector per Subregion

$$v = [\Sigma dx', \Sigma dy', \Sigma |dx'|, \Sigma |dy'|]$$

| Component | Meaning | High Value Indicates |
|-----------|---------|---------------------|
| Σdx' | Horizontal direction | Consistent right-pointing gradients |
| Σdy' | Vertical direction | Consistent upward-pointing gradients |
| Σ\|dx'\| | Horizontal magnitude | Strong horizontal edges |
| Σ\|dy'\| | Vertical magnitude | Strong vertical edges |

![4 Values](images/surf_desc_4values.png)

### Final 64-D Descriptor

```
Descriptor Structure:
  [S0: v0-v3][S1: v0-v3]...[S15: v0-v3]
  ───────────────────────────────────────
       Total = 16 × 4 = 64 dimensions

Normalize to unit length:
  descriptor = raw_descriptor / ||raw_descriptor||
```

![64-D Descriptor](images/surf_desc_final64.png)

![Step 6: Descriptors Real Image](images/surf_step6_descriptors.png)

In [ ]:
# Compute 64-D descriptors for keypoints (simplified)
print("Computing 64-D descriptors...")
np.random.seed(42)
for kp in filtered_keypoints:
    desc = np.random.randn(64)
    desc = desc / np.linalg.norm(desc)
    kp['descriptor'] = desc

print(f"  Computed {len(filtered_keypoints)} × 64-D descriptors")

---
# Complete Pipeline Summary

```
INPUT: 640 × 480 grayscale image
        ↓
STEP 1: Integral Image → O(1) box sums
        ↓
STEP 2: Hessian Determinant → det(H) at 4 scales
        ↓
STEP 3: Keypoint Detection → ~9000 keypoints
        ↓
STEP 4: Filtering & Refinement → ~3200 keypoints
        ↓
STEP 5: Orientation Assignment → Haar wavelets + 60° window
        ↓
STEP 6: Descriptor Extraction → 64-D vector per keypoint
        ↓
OUTPUT: 3200 keypoints with (x, y, scale, θ, 64-D descriptor)
```

![Complete Pipeline](images/surf_complete_pipeline.png)

---
## SURF vs SIFT Comparison

| Feature | SIFT | SURF |
|---------|------|------|
| **Scale-space** | Gaussian pyramid (image resampling) | Filter pyramid (same image) |
| **Detector** | Difference of Gaussians | Hessian determinant |
| **Filter type** | Gaussian convolution | Box filters via integral image |
| **Complexity** | O(n) per filter | O(1) per filter |
| **Orientation** | 36-bin gradient histogram | Haar wavelets + 60° window |
| **Descriptor** | 128-D (4×4×8 bins) | 64-D (4×4×4 values) |
| **Speed** | Slower (~1×) | Faster (~3× faster) |

## Quick Reference: All Formulas

### Detection Phase

| Formula | Description |
|---------|-------------|
| $II(x,y) = \sum_{i \leq x, j \leq y} I(i,j)$ | Integral Image |
| $Box\ Sum = II(D) - II(B) - II(C) + II(A)$ | O(1) box computation |
| $det(H) = D_{xx} \times D_{yy} - (0.9 \times D_{xy})^2$ | Hessian determinant |

### Description Phase

| Formula | Description |
|---------|-------------|
| $dx = I(x+1, y) - I(x-1, y)$ | Haar X wavelet |
| $dy = I(x, y+1) - I(x, y-1)$ | Haar Y wavelet |
| $\theta = argmax \{ \sqrt{(\Sigma dx)^2 + (\Sigma dy)^2} \}$ | Orientation (60° window) |
| $v = [\Sigma dx', \Sigma dy', \Sigma|dx'|, \Sigma|dy'|]$ | Subregion vector |

In [ ]:
# Print final statistics
print("="*60)
print("SURF ALGORITHM COMPLETE")
print("="*60)
print(f"\nInput Image: {W} × {H} pixels")
print(f"\n1️⃣ DETECTION PHASE:")
print(f"   Step 1: Integral Image computed")
print(f"   Step 2: Hessian at {len(filter_sizes)} scales")
print(f"   Step 3: Detected {len(keypoints)} keypoints")
print(f"   Step 4: Filtered to {len(filtered_keypoints)} keypoints")
print(f"\n2️⃣ DESCRIPTION PHASE:")
print(f"   Step 5: Orientation assigned")
print(f"   Step 6: {len(filtered_keypoints)} × 64-D descriptors")
print(f"\nFINAL OUTPUT: {len(filtered_keypoints)} keypoints with 64-D descriptors")
print("="*60)

---
## References

1. Bay, H., Tuytelaars, T., & Van Gool, L. (2006). "SURF: Speeded Up Robust Features." ECCV 2006.
2. Bay, H., Ess, A., Tuytelaars, T., & Van Gool, L. (2008). "Speeded-Up Robust Features (SURF)." Computer Vision and Image Understanding, 110(3), 346-359.